# Features extraction code

## Requirements

In [43]:
!pip3 install pandas openpyxl numpy opensmile torch torchaudio torchcodec

## Imports

In [38]:
import pandas as pd
import openpyxl
import json
import statistics
import numpy as np
import opensmile
import torch
import torchaudio

## Dataframe creation and general variables

In [11]:
features_df = pd.read_excel("../extracted_features/extracted_features.xlsx")

data_path = "../data/"
valid_participants = features_df["participant_id"].unique()

features_df

,participant_id,gender,age,condition,perceived_topic_intimacy,personal_discomfort,revealing_personal_information
0,4,M,23,1,5,1.571,1.25
1,4,M,23,2,4,1.000,1.00
2,5,M,20,1,3,1.143,1.25
3,5,M,20,2,4,2.429,2.00
4,6,F,27,1,4,1.143,1.00
5,6,F,27,2,3,1.286,2.75
6,7,F,21,1,1,1.000,1.75
7,7,F,21,2,3,1.000,1.75
8,9,M,26,1,4,1.714,2.00
9,9,M,26,2,2,1.286,1.50


## General features

### Activity time

In [17]:
activity_time_list = []

for participant in valid_participants:
    for condition in ["c1", "c2"]:
        stroke_file_path = data_path + "p" + str(participant) + "/drawing_data/strokes_" + condition + ".json"
        with open(stroke_file_path, "r") as file:
            data = json.load(file)
        activity_time_list.append(round(data["activity_time"], 2))

features_df["activity_time"] = activity_time_list

## Drawing dynamics features

### Stroke count

In [19]:
stroke_count_list = []

for participant in valid_participants:
    for condition in ["c1", "c2"]:
        stroke_file_path = data_path + "p" + str(participant) + "/drawing_data/strokes_" + condition + ".json"
        with open(stroke_file_path, "r") as file:
            data = json.load(file)
        stroke_count_list.append(len(data["strokes"]) + len(data["undone_erased_strokes"]))

features_df["stroke_count"] = stroke_count_list

### Stroke length (total, mean, std)

In [21]:
def compute_stroke_length(stroke_points):
        point_coordinates = np.array(stroke_points)

        # Differences between consecutive points
        diffs = np.diff(point_coordinates, axis=0)

        # Euclidian distances between consecutive points
        euclidian_distances = np.sqrt((diffs ** 2).sum(axis=1))

        # Sum all the distances between each consecutive points of the stroke
        total_stroke_length = euclidian_distances.sum()

        return total_stroke_length

In [24]:
stroke_total_length_list = []
stroke_mean_length_list = []
stroke_std_length_list = []

for participant in valid_participants:
    for condition in ["c1", "c2"]:
        stroke_file_path = data_path + "p" + str(participant) + "/drawing_data/strokes_" + condition + ".json"
        with open(stroke_file_path, "r") as file:
            data = json.load(file)

        # Extract length of each stroke of participant
        stroke_length_array = []
        for stroke in (data["strokes"] + data["undone_erased_strokes"]):
            stroke_length_array.append(compute_stroke_length(stroke["points"]))

        # Compute total, mean, and std stroke length
        stroke_total_length_list.append(round(sum(stroke_length_array), 2))
        stroke_mean_length_list.append(round(statistics.mean(stroke_length_array), 2))
        stroke_std_length_list.append(round(statistics.stdev(stroke_length_array), 2))

features_df["stroke_total_length"] = stroke_total_length_list
features_df["stroke_mean_length"] = stroke_mean_length_list
features_df["stroke_std_length"] = stroke_std_length_list

### Stroke speed and acceleration (mean and std)

### Stroke pressure (mean, std, slope over time?)

### Temporal drawing behavior

### Editing behavior

## Drawing production features

## Speech dynamics features

In [39]:
# Initialize openSMILE with eGeMAPS functional features
smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.eGeMAPSv02,
    feature_level=opensmile.FeatureLevel.Functionals,
)

features_of_interest = [
    "F0semitoneFrom27.5Hz_sma3nz_amean",
    "F0semitoneFrom27.5Hz_sma3nz_stddevNorm",
    "F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2",
    "loudness_sma3_amean",
    "loudness_sma3_stddevNorm",
    "jitterLocal_sma3nz_amean",
    "shimmerLocaldB_sma3nz_amean",
    "HNRdBACF_sma3nz_amean",
]

In [44]:
# Load Silero VAD
model, utils = torch.hub.load(
    repo_or_dir='snakers4/silero-vad',
    model='silero_vad',
    force_reload=False
)

(get_speech_timestamps,
 save_audio,
 read_audio,
 VADIterator,
 collect_chunks) = utils


def compute_speech_time(audio_path):
    wav, sr = torchaudio.load(audio_path)

    # Ensure mono
    if wav.shape[0] > 1:
        wav = torch.mean(wav, dim=0, keepdim=True)

    # Resample from 44.1 KHz to 16 KHz
    if sr != 16000:
        wav = torchaudio.functional.resample(wav, sr, 16000)
        sr = 16000

    speech_timestamps = get_speech_timestamps(wav.squeeze(0), model, sampling_rate=sr)

    total_speech_time = sum((ts['end'] - ts['start']) for ts in speech_timestamps) / sr

    return total_speech_time

Using cache found in /home/maure/.cache/torch/hub/snakers4_silero-vad_master


In [45]:
# Create storage dictionary
opensmile_data = {feat: [] for feat in features_of_interest}
speech_time_list = []

for participant in valid_participants:
    for condition in ["c1", "c2"]:

        audio_file_path = data_path + "p" + str(participant) + "/audio_data/p" + str(participant) + condition + "_copy.wav"

        # Extract opensmile features
        print("Processing participant:", str(participant), "- condition:", condition)
        result = smile.process_file(audio_file_path)

        # opensmile result is a pandas dataframe (1 row for functionals)
        for feature in features_of_interest:
            value = result[feature].values[0]
            opensmile_data[feature].append(value)

        # Compute speech time
        speech_time = compute_speech_time(audio_file_path)
        speech_time_list.append(speech_time)

# Add opensmile features to dataframe
for feature in features_of_interest:
    features_df[feature] = opensmile_data[feature]

# Add speech time to dataframe
features_df["total_speech_time"] = speech_time_list

# Add speech ratio to dataframe
features_df["speech_ratio"] = (features_df["total_speech_time"] / features_df["activity_time"])

Processing participant: 4 - condition: c1
Processing participant: 4 - condition: c2
Processing participant: 5 - condition: c1
Processing participant: 5 - condition: c2
Processing participant: 6 - condition: c1
Processing participant: 6 - condition: c2
Processing participant: 7 - condition: c1
Processing participant: 7 - condition: c2
Processing participant: 9 - condition: c1
Processing participant: 9 - condition: c2
Processing participant: 11 - condition: c1
Processing participant: 11 - condition: c2
Processing participant: 12 - condition: c1
Processing participant: 12 - condition: c2
Processing participant: 13 - condition: c1
Processing participant: 13 - condition: c2
Processing participant: 14 - condition: c1
Processing participant: 14 - condition: c2
Processing participant: 15 - condition: c1
Processing participant: 15 - condition: c2
Processing participant: 16 - condition: c1
Processing participant: 16 - condition: c2
Processing participant: 17 - condition: c1
Processing participan

KeyError: 'activity_duration'

## Speech production features (transcript)

## Display and save final features dataframe

In [47]:
features_df.to_excel("../extracted_features/extracted_features_final.xlsx", index=False)

features_df

,participant_id,gender,age,condition,perceived_topic_intimacy,personal_discomfort,revealing_personal_information,activity_time,stroke_count,stroke_total_length,...,F0semitoneFrom27.5Hz_sma3nz_amean,F0semitoneFrom27.5Hz_sma3nz_stddevNorm,F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2,loudness_sma3_amean,loudness_sma3_stddevNorm,jitterLocal_sma3nz_amean,shimmerLocaldB_sma3nz_amean,HNRdBACF_sma3nz_amean,total_speech_time,speech_ratio
0,4,M,23,1,5,1.571,1.25,1209.19,464,32406.17,...,22.730339,0.206271,6.394100,0.720653,0.784576,0.052364,1.437832,0.721781,265.572000,0.219628
1,4,M,23,2,4,1.000,1.00,1118.29,311,31294.04,...,22.718616,0.196353,5.203865,0.788069,0.791739,0.045662,1.446904,0.573326,436.644000,0.390457
2,5,M,20,1,3,1.143,1.25,976.01,483,251620.84,...,25.529953,0.167680,4.161392,1.161250,0.698495,0.040078,1.207340,3.940016,441.841250,0.452702
3,5,M,20,2,4,2.429,2.00,1007.23,318,207701.98,...,26.459944,0.172920,5.268013,1.366651,0.710858,0.043603,1.250476,3.951181,538.496000,0.534631
4,6,F,27,1,4,1.143,1.00,849.10,115,29558.12,...,30.298620,0.264035,14.135735,0.593465,0.661590,0.043968,1.169680,4.194973,237.840000,0.280108
5,6,F,27,2,3,1.286,2.75,556.19,137,15112.46,...,30.457277,0.251553,13.270615,0.584809,0.619730,0.041284,1.134220,4.613741,160.780000,0.289074
6,7,F,21,1,1,1.000,1.75,679.11,157,32312.35,...,32.126938,0.183674,3.902401,0.933588,0.607276,0.040284,1.065604,7.012737,348.904000,0.513767
7,7,F,21,2,3,1.000,1.75,508.09,105,18139.80,...,32.114250,0.195904,4.463032,0.878542,0.591010,0.042389,1.115764,6.872468,263.025812,0.517676
8,9,M,26,1,4,1.714,2.00,796.32,106,149419.08,...,23.867699,0.158444,2.880827,0.736282,0.622273,0.036094,1.220165,3.156016,228.916000,0.287467
9,9,M,26,2,2,1.286,1.50,564.09,83,31562.58,...,24.051962,0.174025,3.338686,0.769878,0.652640,0.036670,1.237442,3.040056,154.428000,0.273765
